In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    input_file_name,
    sha2,
    concat_ws,
    to_timestamp,
    year,
    month,
    dayofmonth
)

In [0]:
RAW_PATH = (
    "abfss://raw@stweatherdefaultstorage.dfs.core.windows.net/"
    "weather/"
)

BRONZE_CHECKPOINT = (
    "abfss://checkpoint@stweatherdefaultstorage.dfs.core.windows.net/"
    "weather/bronze/checkpoint/"
)

BRONZE_SCHEMA = (
    "abfss://checkpoint@stweatherdefaultstorage.dfs.core.windows.net/"
    "weather/bronze/schema/"
)

BRONZE_TABLE = (
    "weather_streaming.weather.weather_bronze"
)

In [0]:
raw_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", BRONZE_SCHEMA)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(RAW_PATH)
)

In [0]:
bronze_df = raw_stream.select(
    col("city"),
    col("latitude"),
    col("longitude"),
    col("event_time"),
    col("temperature_c"),
    col("humidity_pct"),
    col("wind_speed_kmh"),
    col("precipitation_mm"),
    col("ingestion_time"),
    col("source"),
    col("pipeline_run_id"),
    col("_metadata.file_path").alias("source_file"),
)

In [0]:
bronze_df = bronze_df.withColumn("event_time", to_timestamp(col("event_time")))
bronze_df = bronze_df.withColumn("bronze_processed_at", current_timestamp()).withColumn(
    "record_hash",
    sha2(
        concat_ws(
            "||", col("city"), col("event_time"), col("latitude"), col("longitude")
        ),
        256,
    ),
)

In [0]:
bronze_df = (
    bronze_df.withColumn("year", year(col("ingestion_time")))
    .withColumn("month", month(col("ingestion_time")))
    .withColumn("day", dayofmonth(col("ingestion_time")))
)

In [0]:
bronze_df = (
    bronze_df.filter(col("city").isNotNull())
    .filter(col("event_time").isNotNull())
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
)

In [0]:
query = (
    bronze_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

In [0]:
query.awaitTermination()